In [1]:
import pandas as pd
import numpy as np

import joblib
from collections import Counter

In [2]:
df = pd.read_csv("../data/processed/final_jobs.csv")

print(df.shape)

df.head()

(97682, 16)


,companyname,title,location,minimumsalary,maximumsalary,salary_gap,avg_experience,aggregaterating,reviewscount,salary_band,experience_category,rating_category,salary_band_encoded,experience_encoded,rating_encoded,tagsandskills
0,Orion,Sr. HR Recruiter (NON IT),Kolkata(Chinar Park),200000.0,400000.0,200000.0,3.0,3.7,965.0,Medium,Mid,Good,2,2,2,"Communication,Manpower,Staffing,Convincing Pow..."
1,"Apollo Hospitals International Limited, Ahmedabad",Fire And Safety Officer,"Gandhinagar, Ahmedabad",300000.0,500000.0,200000.0,8.5,4.0,5162.0,Medium,Expert,Good,2,1,2,"Safety Officer Activities,Fire Protection,Fire..."
2,TVS Credit Services Ltd,Opening For Performance Marketing - Chennai,Chennai,0.0,0.0,0.0,15.0,4.2,2892.0,Unknown,Expert,Excellent,5,1,1,"Performance Marketing,User Acquisition,growth ..."
3,GNR Global Services,Medical Billing Executive,"Mohali, Chandigarh, Kharar, Zirakpur",70000.0,200000.0,130000.0,1.5,3.7,965.0,Low,Entry,Good,1,0,2,"Fluent English,Spoken English,Good English Com..."
4,Cadila Pharmaceuticals,Senior Group Product Manager - CNS Therapy,Ahmedabad,800000.0,1800000.0,1000000.0,7.5,3.4,2134.0,Very High,Senior,Good,4,3,2,"Product Marketing,CNS,Product Management,Nephr..."


In [3]:
df["tagsandskills"].isna().sum()

np.int64(0)

In [4]:
df["tagsandskills"] = df["tagsandskills"].fillna("")

In [5]:
def clean_skills(text):

    if pd.isna(text):
        return []

    skills = str(text).split(",")

    skills = [s.strip().lower() for s in skills]

    return list(set(skills))

In [6]:
df["skill_list"] = df["tagsandskills"].apply(clean_skills)

df[["title","skill_list"]].head()

,title,skill_list
0,Sr. HR Recruiter (NON IT),"[communication skills, staffing, convincing po..."
1,Fire And Safety Officer,"[fire safety, fire engineering, safety managem..."
2,Opening For Performance Marketing - Chennai,"[user acquisition, growth marketing, paid mark..."
3,Medical Billing Executive,"[fluent, medical, english, medical billing, go..."
4,Senior Group Product Manager - CNS Therapy,"[brand marketing, cns, neurology, brand manage..."


In [7]:
skill_dictionary = {}

for _, row in df.iterrows():

    job = row["title"]

    skills = row["skill_list"]

    if job not in skill_dictionary:

        skill_dictionary[job] = Counter()

    skill_dictionary[job].update(skills)

In [8]:
for job in skill_dictionary:

    skill_dictionary[job] = [

        skill

        for skill, count

        in skill_dictionary[job].most_common(20)

    ]

print(len(skill_dictionary))

55079


In [9]:
first_job = list(skill_dictionary.keys())[0]

print(first_job)

print(skill_dictionary[first_job])

Sr. HR Recruiter (NON IT)
['communication skills', 'staffing', 'convincing power', 'hiring', 'manpower', 'recruitment', 'sr', 'communication']


In [10]:
def recommend_skills(current_skills, target_job):

    current = [s.lower().strip() for s in current_skills]

    required = skill_dictionary.get(target_job, [])

    missing = [

        skill

        for skill in required

        if skill not in current

    ]

    match = 100 * (

        len(required) - len(missing)

    ) / max(len(required),1)

    return {

        "Match Score": round(match,2),

        "Required Skills": required,

        "Missing Skills": missing

    }

In [11]:
recommend_skills(

    current_skills=[

        "python",

        "sql",

        "excel"

    ],

    target_job="Data Scientist"

)

{'Match Score': 10.0,
 'Required Skills': ['python',
  'machine learning',
  'natural language processing',
  'data science',
  'data',
  'sql',
  'artificial intelligence',
  'deep learning',
  'algorithms',
  'tensorflow',
  'generative ai',
  'machine',
  'data analysis',
  'science',
  'keras',
  'data scientist',
  'analytical',
  'aws',
  'power bi',
  'pytorch'],
 'Missing Skills': ['machine learning',
  'natural language processing',
  'data science',
  'data',
  'artificial intelligence',
  'deep learning',
  'algorithms',
  'tensorflow',
  'generative ai',
  'machine',
  'data analysis',
  'science',
  'keras',
  'data scientist',
  'analytical',
  'aws',
  'power bi',
  'pytorch']}

In [12]:
job = list(skill_dictionary.keys())[5]

print(job)

recommend_skills(

    ["python","excel"],

    job

)

Senior Accountant


{'Match Score': 0.0,
 'Required Skills': ['accounting',
  'gst',
  'tds',
  'income tax',
  'tds calculation',
  'senior',
  'gst compliance',
  'e way bill',
  'tally erp',
  'accounts finalisation',
  'balance sheet finalisation',
  'accounts reconciliation',
  'remittance processing',
  'tally',
  'bank reconciliation',
  'taxation',
  'gst return',
  'tally prime',
  'financial reporting',
  'tds filing'],
 'Missing Skills': ['accounting',
  'gst',
  'tds',
  'income tax',
  'tds calculation',
  'senior',
  'gst compliance',
  'e way bill',
  'tally erp',
  'accounts finalisation',
  'balance sheet finalisation',
  'accounts reconciliation',
  'remittance processing',
  'tally',
  'bank reconciliation',
  'taxation',
  'gst return',
  'tally prime',
  'financial reporting',
  'tds filing']}

In [13]:
joblib.dump(

    skill_dictionary,

    "../outputs/models/skill_dictionary.pkl"

)

print("Skill Dictionary Saved!")

Skill Dictionary Saved!
